### Coleta de dados de Temperatura, Umidade e Precipitação

Será utilizado o dataset derived-era5-single-levels-daily-statistics do ERA5

Documentação em:
https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics?tab=documentation

Os dados extraídos serão:
<pre>
- Precipitação  -> variável total_precipitation, está variável é retornada em metros, será necessário conveter para milimetros (dividir por 1000)
</pre>

Os dados serão coletados por Ano e Mês 

Os dados requisitados estão no retangulo geográfico geográfico [6, -74, -34, -35] -> [Norte, Oeste, Sul, Leste] em graus onde está o Brasil




In [ ]:
import cdsapi
import os, sys
import xarray as xr
from pyspark.sql import functions as F

In [ ]:
# Cria a conexão Spark

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [ ]:
# Os arquivos utilizados durante o processamento serão removidos no final do notebook
remover_arquivos = []

Requisição dos dados da variável derived-era5-single-levels-daily-statistics do ERA5 utilizando a biblioteca cdsapi

In [ ]:
dataset = "derived-era5-single-levels-daily-statistics"
request = {
    "product_type": "reanalysis",
    "variable": [
        "total_precipitation"
    ],
    "year": "2025",
    "month": ["01"],
    "day": ["01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15"
    ],
    "daily_statistic": "daily_mean",
    "time_zone": "utc-03:00",
    "frequency": "1_hourly",
    "area": [6, -74, -34, -35] # Retangulo definido por [Norte, Oeste, Sul, Leste] em graus onde está o Brasil
}

# Informações de autenticação estão em:
# C:\Users\DRT90628\.ecmwfdatastoresrc
# *** Criar um novo contrato de autenticação deverá ser criado usando um usuário de serviços do Einstein

client = cdsapi.Client(
    url = os.getenv("ECMWF_DATASTORES_URL"),
    key = os.getenv("ECMWF_DATASTORES_KEY"),
)

ret_download = client.retrieve(dataset, request).download()
remover_arquivos.append(ret_download)

print(f"Download completed: {ret_download}")

Esta função ira converter os dados dos arqivos .nc para o format Dask para então converter para Dataframe Spark <br>
Isso deixa o processamento em paralelo e será importante para processamento de grandes volumes (1 ano com todos os meses e dias)

In [ ]:
def convert_netcdf4_Spark(file_name):
    with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Ondas_Calor\\{file_name}"
                        ,engine="netcdf4"
                        ,chunks={"time": 365
                                ,"latitude": 100
                                ,"longitude": 100 }
                        ) as ds:
        
        # Transforma o Dataset em um Spark Dataframe
        df_dask   = ds.to_dask_dataframe()
        df_dask_c = df_dask.compute()
        df_spark  = spark.createDataFrame(df_dask_c)    
    return df_spark

In [ ]:
df_precipitacao = convert_netcdf4_Spark(ret_download)

In [ ]:
drop_cols = ["valid_time", "number", "tp"]
df_precipitacao = \
    (df_precipitacao
        .withColumns({"indicador": F.lit("precipitacao")
                     ,"valor": (F.col("tp") * F.lit(1000)).cast('double')
                     ,"unidade_medida": F.lit("celsius")
                     ,"data_medicao": F.col("valid_time").cast("date")}
                    )
        .drop(*drop_cols)
    )

df_precipitacao.printSchema()
df_precipitacao.show()

In [ ]:

df_precipitacao = \
       (df_precipitacao
            .select("data_medicao"
                   ,"latitude"
                   ,"longitude"
                   ,"indicador"
                   ,"valor"
                   ,"unidade_medida"))

In [ ]:
# df_precipitacao.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_precipitacao.csv", index=False)

df_precipitacao.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_precipitacao.parquet")

In [ ]:
# df_csv = spark.read.csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade_temperatura_precipitacao.csv", header=True, inferSchema=True)
# print("df_csv:", df_csv.count())
# df_csv.printSchema()
# df_csv.show(10,False)


df_parquet = spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_precipitacao.parquet")
print("df_csv:", df_parquet.count())
df_parquet.printSchema()
df_parquet.show(10,False)


In [ ]:
# **** INCLUIR EXCLUSÃO DE ARQUIVOS (.zip e .nc)

for file in remover_arquivos:
    print("Arquivo:", file, end="")
    os.remove(file)
    print(" Removido com sucesso")
